Importar bibliotecas

In [1]:
# Importa a biblioteca Pandas para criação e manipulação de DataFrames.
import pandas as pd

# Importa a biblioteca NumPy para operações numéricas.
import numpy as np

# Importa a biblioteca Joblib para carregar os dados preparados e os modelos treinados.
import joblib

# Importa as métricas utilizadas na avaliação dos modelos.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

Carregar os dados preparados

In [2]:
# Carrega os dados preparados com embeddings e características estruturadas.
dados_preparados_embb = joblib.load(
    "../data/processed/dados_preparados_embb.joblib"
)

# Recupera os dados de teste contendo os embeddings e as características estruturadas.
X_teste_embb = dados_preparados_embb["X_teste_embb"]

# Recupera os valores reais da variável-alvo utilizados na avaliação.
y_teste = dados_preparados_embb["y_teste"]

# Exibe o formato dos dados de teste para confirmar a quantidade de mensagens e características.
print("Formato dos dados de teste:", X_teste_embb.shape)

# Exibe a quantidade de valores reais disponíveis para avaliação.
print("Quantidade de valores reais:", len(y_teste))

Formato dos dados de teste: (270, 408)
Quantidade de valores reais: 270


Carregar os quatro modelos

In [3]:
# Carrega o modelo Logistic Regression treinado com embeddings.
modelo_logistic_regression_embb = joblib.load(
    "../models/modelo_logistic_regression_embb.joblib"
)

# Carrega o modelo LinearSVC calibrado treinado com embeddings.
modelo_linear_svc_embb = joblib.load(
    "../models/modelo_linear_svc_embb.joblib"
)

# Carrega o modelo Random Forest treinado com embeddings.
modelo_random_forest_embb = joblib.load(
    "../models/modelo_random_forest_embb.joblib"
)

# Carrega o modelo XGBoost treinado com embeddings.
modelo_xgboost_embb = joblib.load(
    "../models/modelo_xgboost_embb.joblib"
)

# Exibe uma mensagem confirmando que os quatro modelos foram carregados.
print("Modelos carregados com sucesso!")

Modelos carregados com sucesso!


Gerar as previsões

In [4]:
# Gera as previsões do modelo Logistic Regression para os dados de teste.
previsoes_logistic_regression_embb = modelo_logistic_regression_embb.predict(
    X_teste_embb
)

# Gera as previsões do modelo LinearSVC calibrado para os dados de teste.
previsoes_linear_svc_embb = modelo_linear_svc_embb.predict(
    X_teste_embb
)

# Gera as previsões do modelo Random Forest para os dados de teste.
previsoes_random_forest_embb = modelo_random_forest_embb.predict(
    X_teste_embb
)

# Gera as previsões do modelo XGBoost para os dados de teste.
previsoes_xgboost_embb = modelo_xgboost_embb.predict(
    X_teste_embb
)

# Exibe a quantidade de previsões produzidas por cada modelo.
print("Logistic Regression:", len(previsoes_logistic_regression_embb))

# Exibe a quantidade de previsões produzidas pelo LinearSVC.
print("LinearSVC:", len(previsoes_linear_svc_embb))

# Exibe a quantidade de previsões produzidas pelo Random Forest.
print("Random Forest:", len(previsoes_random_forest_embb))

# Exibe a quantidade de previsões produzidas pelo XGBoost.
print("XGBoost:", len(previsoes_xgboost_embb))

Logistic Regression: 270
LinearSVC: 270
Random Forest: 270
XGBoost: 270


Agora vamos calcular as métricas oficiais da avaliação com embeddings

Como já temos as previsões dos 4 modelos, vamos montar uma tabela consolidada com:

Accuracy
Precision
Recall
F1-score
ROC-AUC

Consolidar as métricas

In [5]:
# Cria uma lista vazia para armazenar as métricas de cada modelo.
resultados_avaliacao_embb = []

# Define os nomes dos quatro modelos avaliados.
nomes_modelos_embb = [
    "Logistic Regression",
    "LinearSVC",
    "Random Forest",
    "XGBoost"
]

# Define as previsões produzidas pelos quatro modelos.
previsoes_modelos_embb = [
    previsoes_logistic_regression_embb,
    previsoes_linear_svc_embb,
    previsoes_random_forest_embb,
    previsoes_xgboost_embb
]

# Define as probabilidades da classe "Possível golpe" produzidas pelos quatro modelos.
probabilidades_modelos_embb = [
    modelo_logistic_regression_embb.predict_proba(X_teste_embb)[:, 1],
    modelo_linear_svc_embb.predict_proba(X_teste_embb)[:, 1],
    modelo_random_forest_embb.predict_proba(X_teste_embb)[:, 1],
    modelo_xgboost_embb.predict_proba(X_teste_embb)[:, 1]
]

# Percorre os quatro modelos, suas previsões e suas probabilidades.
for nome_modelo, previsoes, probabilidades in zip(
    nomes_modelos_embb,
    previsoes_modelos_embb,
    probabilidades_modelos_embb
):

    # Calcula a proporção de classificações corretas realizadas pelo modelo.
    accuracy = accuracy_score(
        y_teste,
        previsoes
    )

    # Calcula a proporção de previsões positivas que realmente eram possíveis golpes.
    precision = precision_score(
        y_teste,
        previsoes
    )

    # Calcula a proporção de possíveis golpes que foram identificados corretamente.
    recall = recall_score(
        y_teste,
        previsoes
    )

    # Calcula a média harmônica entre Precision e Recall.
    f1 = f1_score(
        y_teste,
        previsoes
    )

    # Calcula a capacidade do modelo de separar as duas classes considerando diferentes thresholds.
    auc = roc_auc_score(
        y_teste,
        probabilidades
    )

    # Adiciona todas as métricas calculadas à lista de resultados.
    resultados_avaliacao_embb.append([
        accuracy,
        precision,
        recall,
        f1,
        auc
    ])

# Cria um DataFrame contendo todas as métricas dos modelos.
df_avaliacao_embb = pd.DataFrame(
    resultados_avaliacao_embb,
    columns=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    index=nomes_modelos_embb
)

# Exibe a tabela consolidada de avaliação dos modelos com embeddings.
df_avaliacao_embb

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Logistic Regression,0.959259,0.969388,0.922330,0.945274,0.992966
LinearSVC,0.962963,0.960396,0.941748,0.950980,0.988547
Random Forest,0.944444,0.940000,0.912621,0.926108,0.989623
XGBoost,0.970370,0.970297,0.951456,0.960784,0.996919


Alterar o threshold do XGBoost com embeddings.

In [6]:
# Obtém as probabilidades de possível golpe produzidas pelo XGBoost com embeddings.
probabilidades_xgboost_embb = modelo_xgboost_embb.predict_proba(X_teste_embb)[:, 1]

# Define os thresholds que serão avaliados.
thresholds_embb = [0.30, 0.40, 0.50, 0.60, 0.70]

# Cria uma lista vazia para armazenar os resultados de cada threshold.
resultados_threshold_embb = []

# Percorre cada threshold definido para o experimento.
for threshold in thresholds_embb:

    # Converte as probabilidades em classes utilizando o threshold atual.
    previsoes_threshold = (probabilidades_xgboost_embb >= threshold).astype(int)

    # Calcula a quantidade de falsos positivos.
    falso_positivo = ((y_teste == 0) & (previsoes_threshold == 1)).sum()

    # Calcula a quantidade de falsos negativos.
    falso_negativo = ((y_teste == 1) & (previsoes_threshold == 0)).sum()

    # Calcula a precisão do modelo no threshold atual.
    precision = precision_score(y_teste, previsoes_threshold)

    # Calcula o recall do modelo no threshold atual.
    recall = recall_score(y_teste, previsoes_threshold)

    # Calcula o F1-score do modelo no threshold atual.
    f1 = f1_score(y_teste, previsoes_threshold)

    # Calcula a acurácia do modelo no threshold atual.
    accuracy = accuracy_score(y_teste, previsoes_threshold)

    # Armazena os resultados do threshold atual.
    resultados_threshold_embb.append([
        threshold,
        accuracy,
        precision,
        recall,
        f1,
        falso_positivo,
        falso_negativo
    ])

# Cria um DataFrame com os resultados dos thresholds.
df_threshold_xgboost_embb = pd.DataFrame(
    resultados_threshold_embb,
    columns=[
        "Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "FP",
        "FN"
    ]
)

# Exibe os resultados dos thresholds do XGBoost com embeddings.
df_threshold_xgboost_embb

,Threshold,Accuracy,Precision,Recall,F1-score,FP,FN
0,0.3,0.966667,0.960784,0.951456,0.956098,4,5
1,0.4,0.970370,0.970297,0.951456,0.960784,3,5
2,0.5,0.970370,0.970297,0.951456,0.960784,3,5
3,0.6,0.966667,0.970000,0.941748,0.955665,3,6
4,0.7,0.970370,0.989691,0.932039,0.960000,1,7


Observe uma coisa interessante: 40% e 50% produziram exatamente os mesmos resultados. Isso significa que nenhuma probabilidade do conjunto de teste está entre 0,40 e 0,50 de forma a alterar a classificação.

Qual threshold escolher?

Para o projeto, eu escolheria 40% ou 50%, e há um motivo importante.

O objetivo é identificar possíveis golpes. Portanto, não queremos aumentar muito o FN (falso negativo).

Comparando:

30%: FN = 5
40%: FN = 5
50%: FN = 5
60%: FN = 6
70%: FN = 7

Minha escolha: 40%

Por quê?

| Critério  | Threshold 40% |
| --------- | ------------: |
| Accuracy  |    **97,04%** |
| Precision |    **97,03%** |
| Recall    |    **95,15%** |
| F1-score  |    **96,08%** |
| FP        |         **3** |
| FN        |         **5** |


Agora vem a comparação mais importante: Comparar o melhor resultado do XGBoost com TF-IDF contra o XGBoost com embeddings.

| Métrica   | XGBoost + TF-IDF | XGBoost + Embeddings |
| --------- | ---------------: | -------------------: |
| ROC-AUC   |         0,995349 |         **0,996919** |
| Precision |           96,19% |           **97,03%** |
| Recall    |       **98,06%** |               95,15% |
| F1-score  |       **97,12%** |               96,08% |
| FP        |                4 |                **3** |
| FN        |            **2** |                    5 |


A comparação mostrou que as duas abordagens apresentaram excelente desempenho. Apesar dos embeddings obterem um ROC-AUC ligeiramente superior, o **XGBoost com TF-IDF apresentou melhor resultado para o objetivo principal do projeto: detectar possíveis golpes e reduzir falsos negativos**.

| Métrica          | XGBoost + TF-IDF (30%) | XGBoost + Embeddings (40%) |
| ---------------- | ---------------------: | -------------------------: |
| Accuracy         |             **97,41%** |                     97,04% |
| Precision        |                 96,19% |                 **97,03%** |
| Recall           |             **98,06%** |                     95,15% |
| F1-score         |             **97,12%** |                     96,08% |
| ROC-AUC          |                 0,9953 |                 **0,9969** |
| Falsos positivos |                      4 |                      **3** |
| Falsos negativos |                  **2** |                          5 |

O **TF-IDF foi escolhido** porque identificou **101 dos 103 possíveis golpes**, deixando apenas **2 casos não detectados**, contra 5 utilizando embeddings. Portanto, mesmo com uma pequena vantagem dos embeddings no ROC-AUC, o **TF-IDF apresentou melhor desempenho na aplicação prática do projeto**.
